# OASIS-2 Multimodal Training Pipeline

This notebook is the entry point for the **Multimodal Alzheimer's AI Monitoring Platform**.
It fuses 3D brain MRI scans with clinical metadata (Age, Sex, MMSE) using the `OASISMultimodalDenseNet` architecture.

In [10]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

# Constants for your environment
DRIVE_ROOT = Path('/content/drive/MyDrive/Cerebrasensecloud')
OASIS2_BUNDLE_ROOT = DRIVE_ROOT / 'OASIS-2'
RUNTIME_ROOT = DRIVE_ROOT / 'backend_runtime'
RUN_NAME = 'oasis2_multimodal_v1' # Fresh multimodal experiment

for name, path in {
    'DRIVE_ROOT': DRIVE_ROOT,
    'OASIS2_BUNDLE_ROOT': OASIS2_BUNDLE_ROOT,
    'RUNTIME_ROOT': RUNTIME_ROOT,
}.items():
    print(f"{name}: {'[OK]' if path.exists() else '[MISSING]'} {path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DRIVE_ROOT: [OK] /content/drive/MyDrive/Cerebrasensecloud
OASIS2_BUNDLE_ROOT: [OK] /content/drive/MyDrive/Cerebrasensecloud/OASIS-2
RUNTIME_ROOT: [OK] /content/drive/MyDrive/Cerebrasensecloud/backend_runtime


In [11]:
import shutil
import subprocess
from pathlib import Path

REPO_ROOT = Path('/content/cerebrasense')
BACKEND_ROOT = REPO_ROOT / 'alz_backend'
REPO_URL = 'https://github.com/Billrichard209/cerebrasense.git'
REQUIRED_COMMIT = '95132f8' # Latest optimized commit

def run_checked(cmd, *, cwd=None, label=None):
    print(f"RUNNING {label or cmd[0]}: {' '.join(cmd)}", flush=True)
    completed = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout, flush=True)
    if completed.stderr:
        print(completed.stderr, flush=True)
    if completed.returncode != 0:
        raise RuntimeError(f"Command failed ({label or cmd[0]}): {' '.join(cmd)}")
    return completed

# Clean up any stale repo clones
for stale_root in [Path('/content/cerebrasense'), Path('/content/Cerebrasense-')]:
    if stale_root.exists():
        shutil.rmtree(stale_root)

run_checked(['git', 'clone', REPO_URL, str(REPO_ROOT)], cwd='/content', label='git-clone')
run_checked(['git', 'checkout', 'main'], cwd=REPO_ROOT, label='git-checkout-main')
run_checked(['python3', '-m', 'pip', 'install', '-r', str(BACKEND_ROOT / 'requirements-colab.txt')], cwd=REPO_ROOT, label='pip-install')

repo_commit = run_checked(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, label='git-rev-parse').stdout.strip()
print(f'Active commit: {repo_commit}')

RUNNING git-clone: git clone https://github.com/Billrichard209/cerebrasense.git /content/cerebrasense


Cloning into '/content/cerebrasense'...

RUNNING git-checkout-main: git checkout main
Your branch is up to date with 'origin/main'.

Already on 'main'

RUNNING pip-install: python3 -m pip install -r /content/cerebrasense/alz_backend/requirements-colab.txt

RUNNING git-rev-parse: git rev-parse HEAD
d5a3bc3eb181eda0cfad05903803eb4bf11a54b2

Active commit: d5a3bc3eb181eda0cfad05903803eb4bf11a54b2


### Start Multimodal Training
Initiates a 50-epoch training run fusing MRI volumes with clinical metadata (Age, Sex, MMSE).

In [12]:
import os
import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU not detected! Change runtime type to T4 GPU before proceeding.")

os.chdir(BACKEND_ROOT)

!python scripts/train_oasis2_colab.py \
    --project-root {BACKEND_ROOT} \
    --runtime-root {RUNTIME_ROOT} \
    --bundle-root {OASIS2_BUNDLE_ROOT} \
    --run-name {RUN_NAME} \
    --epochs 50 \
    --batch-size 2 \
    --gradient-accumulation-steps 4 \
    --num-workers 2 \
    --image-size 96 96 96 \
    --learning-rate 2e-4 \
    --weight-decay 0.01 \
    --scheduler cosine \
    --weighted-sampling \
    --seed 42 \
    --split-seed 42 \
    --device auto \
    --config configs/oasis2_train.yaml

Starting OASIS-2 Colab bundle pipeline
Resolved OASIS-2 bundle root: /content/drive/MyDrive/Cerebrasensecloud/OASIS-2
Runtime data root: /content/drive/MyDrive/Cerebrasensecloud/backend_runtime/data
Runtime outputs root: /content/drive/MyDrive/Cerebrasensecloud/backend_runtime/outputs
Inspecting uploaded OASIS-2 bundle
Upload bundle inspection status: pass
Resolving OASIS-2 metadata template source
Resolved metadata template source: /content/drive/MyDrive/Cerebrasensecloud/OASIS-2/backend_reference/oasis2_metadata_template.csv
Running preflight runtime refresh from Drive bundle
Refreshing runtime artifacts from source root: /content/drive/MyDrive/Cerebrasensecloud/OASIS-2
Copying OASIS-2 metadata template into runtime data root
Runtime metadata template path: /content/drive/MyDrive/Cerebrasensecloud/backend_runtime/data/interim/oasis2_metadata_template.csv
Runtime metadata template has no labels; resolving official OASIS-2 demographics source
Merging official OASIS-2 demographics from:

### Result Summary
View final metrics after training completes.

In [14]:
import json
import pandas as pd
from pathlib import Path

run_root = RUNTIME_ROOT / 'outputs' / 'runs' / 'oasis2' / RUN_NAME
summary_path = run_root / 'reports' / 'colab_run_summary.json'
metrics_path = run_root / 'metrics' / 'epoch_metrics.csv'

if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print(f"\nTraining Summary for {RUN_NAME}:")
    print(f"- Best Epoch: {summary.get('best_epoch')}")
    print(f"- Best Val AUROC: {summary.get('best_monitor_value'):.4f}")
    print(f"- Best Checkpoint: {summary.get('best_checkpoint')}")

if metrics_path.exists():
    df = pd.read_csv(metrics_path)
    print("\nLast 5 Epochs:")
    print(df.tail())



Training Summary for oasis2_multimodal_v1:
- Best Epoch: 8
- Best Val AUROC: 0.6750
- Best Checkpoint: /content/drive/MyDrive/Cerebrasensecloud/backend_runtime/outputs/runs/oasis2/oasis2_multimodal_v1/checkpoints/best_model.pt

Last 5 Epochs:
    epoch  learning_rate  train_loss  val_loss  accuracy     auroc  precision  \
11     12       0.000141    0.218669  0.214284  0.481481  0.566529   0.483871   
12     13       0.000131    0.212553  0.215975  0.518519  0.613169   0.510638   
13     14       0.000121    0.213659  0.209233  0.537037  0.604938   0.550000   
14     15       0.000110    0.211898  0.208157  0.592593  0.609053   0.592593   
15     16       0.000100    0.213008  0.210208  0.592593  0.592593   0.575758   

      recall        f1  sensitivity  specificity  train_batches  val_batches  
11  0.555556  0.517241     0.555556     0.407407            130           27  
12  0.888889  0.648649     0.888889     0.148148            130           27  
13  0.407407  0.468085     0.407